In [2]:
import pandas as pd
df = pd.read_csv('../data/processed/churn_predictions.csv')
print(df['CLV'].describe())
print(df.nlargest(10, 'CLV')[['CustomerID', 'CLV', 'PredictedChurn']])

count      784.000000
mean      1906.715078
std       5109.989544
min          6.900000
25%        310.875000
50%        679.930000
75%       1678.770000
max      81024.840000
Name: CLV, dtype: float64
     CustomerID       CLV  PredictedChurn
653       16029  81024.84               0
685       15061  54534.14               0
531       15749  44534.30               1
4         13798  37153.85               0
547       16013  37130.60               0
397       12748  33719.73               0
421       14680  28754.11               0
778       13081  28337.38               0
688       13777  25977.16               0
255       17675  20374.28               0


## Day 5 — Threshold Tuning & Business Cost Analysis: Observations

### 1. Threshold Analysis — Finding the Cost-Optimal Cutoff

The default classification threshold (0.5) is not the same as the business-optimal threshold. Using assumed costs of $5 per false positive (wasted win-back offer on a loyal customer) and $50 per false negative (a missed churner, i.e. lost customer), the analysis tested thresholds from 0.10 to 0.90:

**Results:**

| Threshold | Recall | Precision | Total Cost |
|----------:|-------:|----------:|-----------:|
| **0.20 (Optimal)** | **99.2%** | **42.9%** | **$1,820** |
| 0.50 (Default) | 85.4% | 51.4% | $2,955 |
| 0.80 | 0.8% | 100.0% | $12,950 |


Lowering the threshold from 0.5 to 0.2 reduces total business cost by **38%** ($2,955 → $1,820). This makes sense given the cost asymmetry: missing a churner is 10x more expensive than a false alarm, so the cost-optimal strategy is to cast a wider net and accept more false positives in exchange for catching nearly every real churner. This is a good concrete example of why "best F1" (threshold 0.5) and "best business outcome" (threshold 0.2) are two different questions — F1 treats both error types as equally costly, but the real business does not.

**Chosen threshold going forward: 0.20**, saved to `models/optimal_threshold.npy`.

### 2. Revenue-at-Risk Analysis

Using the 0.20 threshold on the test set (784 customers):

- **603 customers** predicted as at-risk of churning
- **Revenue at risk: $559,896.14** (total CLV of all predicted churners)
- **Top 20% highest-risk customers** (156 people) account for **$30,620.54** of   that revenue
- If even **50% of predicted churners were retained**, the model implies a potential revenue save of **$279,948.07**

This is the number that translates the model from "a classifier with 60% PR-AUC" into a concrete business case: acting on these predictions has a quantifiable dollar upside.

This is the number that translates the model from "a classifier with 0.60 PR-AUC" into a concrete business case: acting on these predictions has a quantifiable dollar upside. Surprisingly, already that the top-risk-by-probability group holds a  small share of the total at-risk revenue — this is investigated further below.

### 3. CLV Outlier Investigation

Before trusting the $559,896 figure, the CLV distribution was checked for skew, since Week 1's exploration already found that a handful of customers/products in this dataset are wholesale/bulk buyers rather than typical retail customers (e.g. StockCode 23843, where 98% of its  olume came from a single bulk order week).

**CLV distribution (784 test customers):**
- Mean: $1,906.72 — Median: $679.93 (mean is ~2.8x the median)
- Std deviation ($5,109.99) exceeds the mean — a strong signal of heavy right skew
- Max: $81,024.84, roughly 119x the median customer's CLV

This confirms the same wholesale-buyer pattern identified in Week 1 is present in the customer-level CLV figures, not just at the product level.

**Important caveat on the CLV formula itself:** the current calculation is `CLV = (Monetary / 12) * 12`, which algebraically simplifies to just `Monetary` — total historical spend. This is a simplified placeholder, not a true forward-looking lifetime value projection (which would typically involve retention probability, discount rates, or expected future purchase behavior). Worth being transparent about this limitation rather than presenting CLV as more rigorous than it is.

**Reassuring finding:** despite the skew, spot-checking the 10 highest-CLV customers shows only 1 of them is a predicted churner. The other 9 — including the single largest customer at $81,024.84 — are predicted as NOT churning. This means the $559,896 revenue-at-risk figure is not being artificially inflated by a small number of wholesale outliers; the model's churn predictions and the extreme-value customers are largely independent of each other in this sample.

### 4. Why High-CLV Customers Aren't Flagged as High-Risk

Spot-checking the 10 highest-CLV customers: only 1 of them is a predicted churner. The other 9 — including the single largest customer at $81,024.84 — are predicted as NOT churning.

This isn't coincidental, it's structural. CLV here is just `Monetary`, and `Monetary` is also one of the model's own input features. A customer can only accumulate high total spend through frequent, sustained purchasing over time — the same behavior (high Frequency, long DaysActive) the model reads as "engaged, low churn risk." High value and low predicted risk aren't independent; they emerge from the same underlying behavior.

This confirms the $559,896 revenue-at-risk figure isn't inflated by misclassified outliers. But it also reveals a blind spot: a long-tenured, 
high-value customer whose behavior recently changed might not get flagged quickly, since their features are built from their entire purchase history, which can dilute a recent shift.


### 5. Business Implication: Risk Score Alone May Underweight High-Value Customers

Combined with Section 2's finding — the top 20% highest-risk-by-probability customers hold only $30,620 of the total $559,896 at risk — this suggests ranking customers by churn probability alone isn't the best way to prioritize retention efforts. A more complete approach would combine churn probability *and* CLV, e.g. flagging any customer above a lower risk threshold specifically when their CLV is also high, rather than relying on probability ranking alone.


### 6. Limitation:

The $5 and $50 costs are flat, illustrative assumptions, not derived from the dataset.

**False negative cost ($50):** treats every missed churner as equally costly,but CLV ranges from ~$680 (median) to $80,000+ in this data — missing a $680 customer and an $80,000 customer are not the same loss. A better version would sum actual CLV of the specific customers missed, instead of a flat number × count.

**False positive cost ($5):** this dataset has no real campaign cost data (offer value, email cost, etc.), so unlike the false negative side, this can't be derived from what's available — it would need real business records.

**Net effect:** the false-negative side of this analysis could be made meaningfully more accurate using data already computed in this project (CLV), while the false-positive side would require external cost data not present in the UCI Online Retail dataset. This asymmetry is worth being explicit about — one limitation is fixable with existing data, the other isn't.


### Summary

The chosen threshold (0.20) meaningfully reduces business cost versus the default (0.5), and the resulting revenue-at-risk figure ($559,896) holds up under scrutiny — it isn't an artifact of a few outlier whale customers being misclassified. The CLV formula itself is a known simplification worth revisiting if this project were extended toward a more rigorous CLV model (e.g. incorporating retention curves or purchase frequency decay).